# Exp03-Part2-t1.基于ResNet18网络完成图像分类任务（Paddle）

在本实践中，我们实践一个更通用的图像分类任务。

**图像分类**（Image Classification）是计算机视觉中的一个基础任务，将图像的语义将不同图像划分到不同类别。很多任务也可以转换为图像分类任务。比如人脸检测就是判断一个区域内是否有人脸，可以看作一个二分类的图像分类任务。

这里，我们使用的计算机视觉领域的经典数据集：CIFAR-10数据集，网络为ResNet18模型，损失函数为交叉熵损失，优化器为Adam优化器，评价指标为准确率。

*******

Adam优化器的介绍参考《神经网络与深度学习》第7.2.4.3节。

*******

# 1. 数据处理

## 1.1 数据集介绍

CIFAR-10数据集包含了10种不同的类别、共60,000张图像，其中每个类别的图像都是6000张，图像大小均为$32\times32$像素。CIFAR-10数据集的示例如 **图1** 所示。

<center><img src="https://ai-studio-static-online.cdn.bcebos.com/12cb79a15a3d40fa9b0124dfe4681ab8b60b82d067fb433eb6da600b7c2ec875" width="600" hegiht="" ></center>
<center><br>图1：CIFAR-10数据集示例</br></center>
<br></br>


## 1.2 自定义数据集
### 1.2.1 数据读取

在本实践中，将原始训练集拆分成了train_set、dev_set两个部分，分别包括40 000条和10 000条样本。将data_batch_1到data_batch_4作为训练集，data_batch_5作为验证集，test_batch作为测试集。
最终的数据集构成为：

* 训练集：40 000条样本。
* 验证集：10 000条样本。
* 测试集：10 000条样本。

读取一个batch数据的代码如下所示：

In [ ]:
import os
import pickle
import numpy as np

def load_cifar10_batch(folder_path, batch_id=1, mode='train'):
    if mode == 'test':
        file_path = os.path.join(folder_path, 'test_batch')
    else:
        file_path = os.path.join(folder_path, 'data_batch_'+str(batch_id))

    # 加载数据集文件
    with open(file_path, 'rb') as batch_file:
        batch = pickle.load(batch_file, encoding='latin1')

    imgs = batch['data'].reshape((len(batch['data']), 3, 32, 32)) / 255.
    labels = batch['labels']

    return np.array(imgs, dtype='float32'), np.array(labels)

# 使用您的实际文件路径
folder_path = r'~\anaconda3\数据文件\cifar-10-batches-py'

# 检查路径是否存在
if not os.path.exists(folder_path):
    print(f"错误：路径 {folder_path} 不存在")
    # 列出可能的目录
    parent_dir = r'~\anaconda3\数据文件'
    if os.path.exists(parent_dir):
        print("该目录下的文件：")
        for item in os.listdir(parent_dir):
            print(f"  {item}")
else:
    try:
        imgs_batch, labels_batch = load_cifar10_batch(
            folder_path=folder_path, 
            batch_id=1, 
            mode='train'
        )
        print(f"成功加载批次！图像形状：{imgs_batch.shape}, 标签形状：{labels_batch.shape}")
        print(f"像素值范围：[{imgs_batch.min():.3f}, {imgs_batch.max():.3f}]")
        print(f"前10个标签：{labels_batch[:10]}")
    except Exception as e:
        print(f"加载数据时出错：{e}")
        # 检查目录内容
        print("\n目录内容：")
        for item in os.listdir(folder_path):
            print(f"  {item}")

查看数据的维度：

In [ ]:
#打印一下每个batch中X和y的维度
print ("batch of imgs shape: ",imgs_batch.shape, "batch of labels shape: ", labels_batch.shape)

可视化观察其中的一张样本图像和对应的标签，代码如下所示：

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

image, label = imgs_batch[1], labels_batch[1]
print("The label in the picture is {}".format(label))
plt.figure(figsize=(2, 2))
# print(image.shape)
# print(image.transpose(1,2,0).shape)
plt.imshow(image.transpose(1,2,0)) # numpy数组，https://blog.csdn.net/trayvontang/article/details/108391139
plt.savefig('cnn-car.pdf')

### 1.2.2 构造Dataset类

构造一个CIFAR10Dataset类，其将继承自`paddle.io.DataSet`类，可以逐个数据进行处理。代码实现如下：

In [ ]:
import os
import pickle
import numpy as np
import paddle
import paddle.io as io
from paddle.vision.transforms import Normalize

def load_cifar10_batch(folder_path, batch_id=1, mode='train'):
    if mode == 'test':
        file_path = os.path.join(folder_path, 'test_batch')
    else:
        file_path = os.path.join(folder_path, 'data_batch_'+str(batch_id))

    # 加载数据集文件
    with open(file_path, 'rb') as batch_file:
        batch = pickle.load(batch_file, encoding='latin1')

    imgs = batch['data'].reshape((len(batch['data']), 3, 32, 32)) / 255.
    labels = batch['labels']

    return np.array(imgs, dtype='float32'), np.array(labels)

class CIFAR10Dataset(io.Dataset):
    def __init__(self, folder_path='datasets/cifar-10-batches-py', mode='train'):
        if mode == 'train':
            # 加载batch1-batch4作为训练集
            self.imgs, self.labels = load_cifar10_batch(folder_path=folder_path, batch_id=1, mode='train')
            for i in range(2, 5):
                imgs_batch, labels_batch = load_cifar10_batch(folder_path=folder_path, batch_id=i, mode='train')
                self.imgs = np.concatenate([self.imgs, imgs_batch])
                self.labels = np.concatenate([self.labels, labels_batch])
        elif mode == 'dev':
            # 加载batch5作为验证集
            self.imgs, self.labels = load_cifar10_batch(folder_path=folder_path, batch_id=5, mode='train')
        elif mode == 'test':
            # 加载测试集
            self.imgs, self.labels = load_cifar10_batch(folder_path=folder_path, mode='test')
        
        # 应用归一化
        self.transform = Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010], data_format='CHW')

    def __getitem__(self, idx):
        img, label = self.imgs[idx], self.labels[idx]
        img = self.transform(img)
        return img, label

    def __len__(self):
        return len(self.imgs)

# 设置数据路径 - 使用您的实际路径
folder_path = r'~\anaconda3\数据文件\cifar-10-batches-py'

# 检查路径是否存在
if not os.path.exists(folder_path):
    print(f"错误：路径 {folder_path} 不存在")
    # 尝试查找其他可能的路径
    parent_dir = r'~\anaconda3\数据文件'
    if os.path.exists(parent_dir):
        print("该目录下的文件：")
        for item in os.listdir(parent_dir):
            print(f"  {item}")
else:
    # 设置随机种子并创建数据集
    paddle.seed(100)
    
    try:
        train_dataset = CIFAR10Dataset(folder_path=folder_path, mode='train')
        dev_dataset = CIFAR10Dataset(folder_path=folder_path, mode='dev')
        test_dataset = CIFAR10Dataset(folder_path=folder_path, mode='test')
        
        print("数据集创建成功！")
        print(f"训练集大小: {len(train_dataset)}")
        print(f"验证集大小: {len(dev_dataset)}")
        print(f"测试集大小: {len(test_dataset)}")
        
        # 测试一个样本
        sample_img, sample_label = train_dataset[0]
        print(f"样本图像形状: {sample_img.shape}")
        print(f"样本标签: {sample_label}")
        
    except Exception as e:
        print(f"创建数据集时出错: {e}")
        print("\n检查目录内容：")
        for item in os.listdir(folder_path):
            print(f"  {item}")

## 1.3 paddle内置数据集（本实践中不使用）

paddle提供API完成内置数据集的下载并为后续的训练任务准备好数据迭代器。</br>
__此处仅做简要介绍，本实践使用1.2节中的自定义数据集，实际应用中都需要自定义数据集。__

In [ ]:
print('飞桨内置数据集：', paddle.vision.datasets.__all__)

**准备训练集和测试集**

准备训练集cifar10_train和测试集cifar10_test

`paddle.vision.datasets.Cifar10()`中的mode='train'和mode='test'分别用于获取Cifar10训练集和测试集

规范化： 通过规范化手段，把输入图像的分布改变成均值为0，标准差为1的标准正态分布，使得最优解的寻优过程明显会变得平缓，训练过程更容易收敛。
*******

在飞桨中，提供了部分视觉领域的高层API，可以直接调用API实现简单的图像处理操作。通过调用`paddle.vision.transforms.Resize`调整大小；调用`paddle.vision.transforms.Normalize`进行标准化处理；使用`paddle.vision.transforms.Compose`将两个预处理操作进行拼接。

*******

In [ ]:
import paddle
import paddle.nn.functional as F
from paddle.vision.transforms import ToTensor
# transform = ToTensor()这一行代码创建了一个ToTensor对象，这是一个数据预处理转换操作。
# ToTensor操作的作用是将PIL图像或NumPy数组转换成PyTorch张量，这是深度学习模型可以接受的输入格式。
# 这个转换通常还会包括归一化等预处理步骤，使得数据更适合用于训练神经网络。
transform = ToTensor() 

cifar10_train = paddle.vision.datasets.Cifar10(mode='train', transform=transform)      # 加载CIFAR-10数据集的训练部分
cifar10_test = paddle.vision.datasets.Cifar10(mode='test', transform=transform)    # 加载CIFAR-10数据集的测试部分

# 2. 模型构建

使用飞桨高层API中的Resnet18进行图像分类实践。

In [ ]:
from paddle.vision.models import resnet18

resnet18_model = resnet18(num_classes=10)  # 默认为3通道

******

飞桨高层 API是对飞桨API的进一步封装与升级，提供了更加简洁易用的API，进一步提升了飞桨的易学易用性。其中，飞桨高层API封装了以下模块：
1. Model类，支持仅用几行代码完成模型的训练；
1. 图像预处理模块，包含数十种数据处理函数，基本涵盖了常用的数据处理、数据增强方法；
1. 计算机视觉领域和自然语言处理领域的常用模型，包括但不限于mobilenet、resnet、yolov3、cyclegan、bert、transformer、seq2seq等等，同时发布了对应模型的预训练模型，可以直接使用这些模型或者在此基础上完成二次开发。
  
飞桨高层 API主要包含在`paddle.vision`和`paddle.text`目录中。

******

# 3. 模型训练

接下来，用一个循环来进行模型的训练，将会:
* 使用 `paddle.io.DataLoader` 来加载数据并组建batch。
* 使用 `F.cross_entropy` 来计算损失值。
* 使用 `paddle.optimizer.Adam` 优化器来进行优化。

在实践中，保存准确率最高的模型作为最佳模型。

用DataLoader对数据集进行封装

In [ ]:
#批次大小
batch_size = 64 
#加载数据
train_loader = io.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dev_loader = io.DataLoader(dev_dataset, batch_size=batch_size)
test_loader = io.DataLoader(test_dataset, batch_size=batch_size) 

In [ ]:
import paddle.nn.functional as F
import paddle.optimizer as opt

#指定运行设备
use_gpu = True if paddle.get_device().startswith("gpu") else False
if use_gpu:
    paddle.set_device('gpu:0')

#定义网络
model = resnet18_model

#学习率大小
lr = 0.001  
#定义优化器，这里使用Adam优化器以及l2正则化策略，相关内容在7.3.3.2和7.6.2中会进行详细介绍
optimizer = opt.Adam(learning_rate=lr, parameters=model.parameters(), weight_decay=0.005)
#定义损失函数
loss_fn = F.cross_entropy

# 初始化记录列表
train_losses = []        # 记录每训练1个epoch后，模型在整个训练集上的平均损失
train_step_losses = []   # 记录每训练1个step后，模型在该batch训练数据上的平均损失
val_losses = []          # 记录模型在验证集上的平均损失        
val_accuracies = []      # 记录模型在验证集上的精度

# 训练模型：使用训练集和验证集进行模型训练，共训练20个epoch。保存准确率最高的模型作为最佳模型。
num_epochs = 20
best_accuracy = 0.0
best_model_state = None
# 模型保存路径
model_path = "best_model.pdparams"             

使用训练集和验证集进行模型训练，共训练20个epoch。在实践中，保存准确率最高的模型作为最佳模型。代码实现如下：

In [ ]:
global_step = 0
# 训练总的步数
num_training_steps = num_epochs * len(train_loader)   # len(train_loader)是batch的数量
log_steps = 500      # 每log_steps个step输出一次训练信息
eval_steps = 500     # 每eval_steps个step输出一次验证信息
# 启动训练
for epoch in range(num_epochs):
    # 训练阶段
    model.train()
    train_epoch_loss = 0.0
    for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
        optimizer.clear_grad()
        predictions = model(X_batch)
        y_batch = paddle.cast(y_batch, dtype='int64')
        train_loss = loss_fn(predictions, y_batch)
#         train_step_losses.append(train_loss.item())  # 记录每个step的损失
        train_step_losses.append((global_step,train_loss.item())) # 记录每个step及损失
        train_loss.backward()
        optimizer.step()
        train_epoch_loss += train_loss.item()
        if global_step % log_steps==0:   # 每训练log_steps个step输出一下信息
            print(f"[Train] epoch: {epoch}/{num_epochs}, step: {global_step}/{num_training_steps}, loss: {train_loss.item():.5f}")

        if eval_steps>0 and global_step>0 and (global_step%eval_steps == 0 or global_step==(num_training_steps-1)):
            # 验证阶段
            model.eval()
            val_epoch_loss = 0.0
            correct = 0
            total = 0
            with paddle.no_grad():
                for X_batch, y_batch in dev_loader:
                    val_predictions = model(X_batch)
                    y_batch = paddle.cast(y_batch, dtype='int64')
                    val_loss = loss_fn(val_predictions, y_batch)
                    val_epoch_loss += val_loss.item()
                    val_pred_class = paddle.argmax(val_predictions, axis=1)
                    correct += (val_pred_class == y_batch).sum().item()
                    total += len(y_batch)

            # 计算平均验证损失和准确率
            val_epoch_loss /= len(dev_loader)
            val_accuracy = correct / total
#             val_losses.append(val_epoch_loss)
            val_losses.append((global_step, val_epoch_loss))   # 记录当前的step和验证集loss
            val_accuracies.append(val_accuracy) 
            if global_step % eval_steps==0:   # 每训练eval_steps个step输出一下验证信息 
                print(f"[Evaluate]  val score: {val_accuracy:.5f}, val loss: {val_epoch_loss:.5f}") 

            # 保存最佳模型
            if val_accuracy > best_accuracy:
                best_model_state = model.state_dict()
                print(f"[Evaluate] best accuracy performence has been updated: {best_accuracy:.5f} --> {val_accuracy:.5f}")
                best_accuracy = val_accuracy
                best_model_state = model.state_dict()
                paddle.save(best_model_state, model_path)
        
        global_step += 1
    
    # 计算平均训练损失
    train_epoch_loss /= len(train_loader)
    train_losses.append(train_epoch_loss)

    

可视化观察训练集与验证集的准确率及损失变化情况。

In [ ]:
import matplotlib.pyplot as plt

# 绘制损失变化曲线
plt.figure(figsize=(10, 5))
# 抽样显示每20个数据点中的第一个数据点
train_items = train_step_losses[::20]
train_steps=[x[0] for x in train_items]   # 获得step
train_losses = [x[1] for x in train_items]  # 获得训练集loss
# 绘制训练集上损失变化曲线
plt.plot(train_steps, train_losses, color="#e4007f", linestyle="-", label='Train Loss')
# 绘制验证集上损失变化曲线
if len(val_losses)>0:
    val_steps=[x[0] for x in val_losses]
    val_losses = [x[1] for x in val_losses]
    plt.plot(val_steps, val_losses, color='#f19ec2', linestyle="--", label='Val Loss')

plt.title('Training and Validation Loss Over Steps')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.legend()
plt.show()

# 绘制评价准确率（验证集上准确率）变化曲线
if len(val_accuracies)>0:
    plt.plot(val_steps, val_accuracies, color='#f19ec2', linestyle="--", label='Validation Accuracy')
    # 绘制坐标轴和图例
    plt.title('Validation Accuracy Over Steps')
    plt.xlabel('Step')
    plt.ylabel('Accuracy')
    plt.legend()
    # 设置Y轴的范围从0到1
    plt.ylim(0, 1.2)

plt.show()

*******

在本实践中，使用了第7章中介绍的Adam优化器进行网络优化，如果使用SGD优化器，会造成过拟合的现象，在验证集上无法得到很好的收敛效果。可以尝试使用第7章中其他优化策略调整训练配置，达到更高的模型精度。

*******

# 4. 模型评价

使用测试数据对在训练过程中保存的最佳模型进行评价，观察模型在测试集上的准确率以及损失情况。代码实现如下：

In [ ]:
import os
# 加载最佳模型
if os.path.exists(model_path):
    model.load_dict(paddle.load(model_path))
    
# 测试模型
model.eval()
test_epoch_loss = 0.0
correct = 0
total = 0
with paddle.no_grad():
    for X_batch, y_batch in test_loader:
        test_predictions = model(X_batch)
        y_batch = paddle.cast(y_batch, dtype='int64')
        test_loss = loss_fn(test_predictions, y_batch)
        test_epoch_loss += test_loss.item()
        test_pred_class = paddle.argmax(test_predictions, axis=1)
#         print("test_pred_class:", test_pred_class)
#         print("labels:", y_batch)
        correct += paddle.sum(test_pred_class == y_batch).item()
        total += len(y_batch)
    
# 计算平均测试损失和准确率
test_epoch_loss /= len(test_loader)
test_accuracy = correct / total
print(f"Test Accuracy: {test_accuracy:.2f}, Test Loss: {test_epoch_loss:.4f}")

# 5. 模型预测

同样地，也可以使用保存好的模型，对测试集中的数据进行模型预测，观察模型效果，具体代码实现如下：

In [ ]:
#获取测试集中的一个batch的数据
X, label = next(test_loader()) 
logits = model(X)
#多分类，使用softmax计算预测概率
pred = F.softmax(logits)
#获取概率最大的类别
pred_class = paddle.argmax(pred[2]).numpy()
label = label[2][0].numpy()
#输出真实类别与预测类别
print("The true category is {} and the predicted category is {}".format(label[0], pred_class[0]))
#可视化图片
plt.figure(figsize=(2, 2))
imgs, labels = load_cifar10_batch(folder_path='datasets/cifar-10-batches-py', mode='test')
plt.imshow(imgs[2].transpose(1,2,0))
plt.savefig('cnn-test-vis.pdf')

# 6. 实践拓展

* 尝试加深残差网络的层数或使用《神经网络与深度学习》中介绍的其他模型完成基于CIFAR10的图像分类实践，观察是否能够得到更高的精度；
* 尝试使用CIFAR-100进行实践，观察不同模型在不同数据集上的学习效果；